## Implementing the Fan Out Pattern

# Implementing the Fan-Out Messaging Pattern in Google Cloud Pub/Sub

Welcome back! In this lesson, we will explore how to implement the **fan-out messaging pattern** using Google Cloud messaging services.

The fan-out pattern allows a single publisher to broadcast messages to multiple independent subscriber systems simultaneously. This architectural blueprint enables highly efficient, decoupled, and scalable communication across microservices. By leveraging a single central topic paired with multiple independent subscriptions, you ensure that every downstream consumer safely receives its own dedicated copy of every single message published.

---

## The System Architecture

Consider a standard system design scenario consisting of three isolated microservices: **Service A**, **Service B**, and **Service C**.

Service A acts as the core event driver and needs to distribute system updates to both Service B and Service C. Instead of requiring Service A to make separate, fragile HTTP connections directly to each target machine, it sends a single request to a centralized Pub/Sub topic channel. Both Service B and Service C hook into this channel via their own independent subscription queues, insulating them from downstream latency drops or sudden processing bottlenecks.

---

## 1. Publishing to a Topic (Service A)

Let's look at how Service A authenticates and dispatches message payloads into the ingest infrastructure. We will also include custom string metadata attributes to provide enrichment context for downstream processing filters.

```python
from google.cloud import pubsub_v1

# Initialize the network publisher client
publisher = pubsub_v1.PublisherClient()

# Define structural project coordinates
project_id = "your-gcp-project-id"
topic_id = "service-a-updates"
topic_path = publisher.topic_path(project_id, topic_id)

# Provision the topic infrastructure safely using a block handler
try:
    publisher.create_topic(request={"name": topic_path})
    print(f"Topic created: {topic_path}")
except Exception:
    print(f"Topic already exists: {topic_path}")

# A. Publish a basic standard message payload (Must be encoded as raw bytes)
future = publisher.publish(topic_path, b"Updates for Services B and C")
print(f"Published message ID: {future.result()}")

# B. Publish an advanced message payload complete with contextual key-value attributes
future_advanced = publisher.publish(
    topic_path,
    b"News update",
    priority="high",
    department="IT"
)
print(f"Published advanced message ID: {future_advanced.result()}")

```

**Console Output:**

> `Topic created: projects/your-gcp-project-id/topics/service-a-updates`
> `Published message ID: 8234567890123456789`
> `Published advanced message ID: 8234567890123456790`

> 💡 **Metadata Filtering Note:** The advanced attributes (`priority` and `department`) journey along with the message shell. Downstream worker nodes can inspect these flags or apply server-side **Filter Expressions** to selectively ingest or skip message packets without reading the binary core body structure.

---

## 2. Creating Multiple Subscriptions for Fan-Out

To realize true fan-out delivery, we attach independent subscriptions to our central topic. Each subscription acts as a separate message buffer queue, allowing individual application backends to consume messages at their own optimized pace.

```python
from google.cloud import pubsub_v1

# Initialize the subscriber consumer client interface
subscriber = pubsub_v1.SubscriberClient()

# Establish unique distinct subscription tracking routes
subscription_b_id = "service-b-sub"
subscription_c_id = "service-c-sub"

subscription_b_path = subscriber.subscription_path(project_id, subscription_b_id)
subscription_c_path = subscriber.subscription_path(project_id, subscription_c_id)

# Loop and safely attach both subscribers to the single ingestion stream path
for subscription_path in [subscription_b_path, subscription_c_path]:
    try:
        subscriber.create_subscription(
            request={"name": subscription_path, "topic": topic_path}
        )
        print(f"Subscription created: {subscription_path}")
    except Exception:
        print(f"Subscription already exists: {subscription_path}")

print("Both subscriptions are ready to receive messages!\n")

```

**Console Output:**

> `Subscription created: projects/your-gcp-project-id/subscriptions/service-b-sub`
> `Subscription created: projects/your-gcp-project-id/subscriptions/service-c-sub`
> `Both subscriptions are ready to receive messages!`

---

## 3. Consuming the Split Event Streams (Services B & C)

Now, let's observe how both consumers pull data down from their personal queues. Notice that because their pipelines are isolated, they can read the exact same message footprints concurrently without causing collisions.

### Processing the Buffer for Service B

```python
# Synchronously pull a chunk of messages for Service B
response = subscriber.pull(
    request={
        "subscription": subscription_b_path,
        "max_messages": 10,
    }
)

print(f"Pulled {len(response.received_messages)} messages for Service B:")
for received_message in response.received_messages:
    message = received_message.message
    print(f"Message ID: {message.message_id}")
    print(f"Data: {message.data.decode('utf-8')}")
    print(f"Attributes: {dict(message.attributes)}")
    print(f"Publish time: {message.publish_time}")
    print("---")
        
    # Acknowledge completion back to the server to drop the message from queue B
    subscriber.acknowledge(
        request={
            "subscription": subscription_b_path,
            "ack_ids": [received_message.ack_id],
        }
    )
print("All messages acknowledged for Service B\n")

```

### Processing the Buffer for Service C

```python
# Pull the message pool for Service C to demonstrate decoupled fan-out operations
response_c = subscriber.pull(
    request={
        "subscription": subscription_c_path,
        "max_messages": 10,
    }
)

print(f"Pulled {len(response_c.received_messages)} messages for Service C:")
for received_message in response_c.received_messages:
    message = received_message.message
    print(f"Message ID: {message.message_id}")
    print(f"Data: {message.data.decode('utf-8')}")
    print(f"Attributes: {dict(message.attributes)}")
    print("---")
        
    # Acknowledge completion back to the server to drop the message from queue C
    subscriber.acknowledge(
        request={
            "subscription": subscription_c_path,
            "ack_ids": [received_message.ack_id],
        }
    )
print("All messages acknowledged for Service C")

```

**Downstream Console Output:**

```text
Pulled 2 messages for Service B:
Message ID: 8234567890123456789
Data: Updates for Services B and C
Attributes: {}
Publish time: 2026-05-25 10:30:45.123456+00:00
---
Message ID: 8234567890123456790
Data: News update
Attributes: {'priority': 'high', 'department': 'IT'}
Publish time: 2026-05-25 10:30:46.789012+00:00
---
All messages acknowledged for Service B

Pulled 2 messages for Service C:
Message ID: 8234567890123456789
Data: Updates for Services B and C
Attributes: {}
---
Message ID: 8234567890123456790
Data: News update
Attributes: {'priority': 'high', 'department': 'IT'}
---
All messages acknowledged for Service C

```

> 🎯 **Architectural Proof:** Notice that the `Message ID` identifiers match exactly across both services (`8234567890123456789` and `8234567890123456790`). This confirms that Pub/Sub handled cloning and distributed duplicates across our infrastructure streams.

---

## Summary

In this lesson, we implemented the **fan-out messaging pattern** using Google Cloud Pub/Sub. By positioning a central topic channel ahead of decoupled subscriptions, a single event payload emitted from Service A safely triggers parallel business operations down in Service B and Service C without any network cross-contamination.

Experiment with configuring additional topics, applying unique routing keys, and managing dead-letter queues to build highly resilient, loosely coupled microservice architectures!

## Building a Messaging System with Google Cloud Pub/Sub

In this task, you'll interactively learn how to use Google Cloud Pub/Sub to build a messaging system. Your task involves running a provided Python script that accomplishes the following:

    Establishes connections to Pub/Sub using the google-cloud-pubsub client library.
    Creates a Pub/Sub topic and subscription.
    Sends a message to the topic.
    Retrieves and prints messages from the subscription.

Important Note: Running scripts can modify the resources in our Google Cloud simulator. To revert to the initial state, you can use the reset button located in the top right corner. However, keep in mind that resetting will erase any code changes. To preserve your code during a reset, consider copying it to the clipboard.

```python
from google.cloud import pubsub_v1

# Create publisher and subscriber clients
publisher = pubsub_v1.PublisherClient()
subscriber = pubsub_v1.SubscriberClient()

# Define project and topic/subscription names
project_id = "your-project-id"  # This would be configured in real environment
topic_name = "MyDemoTopic"
subscription_name = "MyDemoSubscription"

# Create topic path
topic_path = publisher.topic_path(project_id, topic_name)

# Create the Pub/Sub topic
topic = publisher.create_topic(request={"name": topic_path})

# Create subscription path
subscription_path = subscriber.subscription_path(project_id, subscription_name)

# Create a subscription to the topic
subscription = subscriber.create_subscription(
    request={"name": subscription_path, "topic": topic_path}
)

# Publish a message to the topic
message_data = "Hello from MyDemoTopic!"
message_bytes = message_data.encode("utf-8")
future = publisher.publish(topic_path, message_bytes)
message_id = future.result()

# Pull messages from the subscription
response = subscriber.pull(
    request={"subscription": subscription_path, "max_messages": 10}
)

# Print out the messages
for received_message in response.received_messages:
    print(received_message.message.data.decode("utf-8"))
    # Acknowledge the message
    subscriber.acknowledge(
        request={"subscription": subscription_path, "ack_ids": [received_message.ack_id]}
    )

```

## Creating Google Cloud Pub/Sub Topics and Subscriptions

In this task, you'll learn how to connect Google Cloud Pub/Sub topics and subscriptions to efficiently manage messaging. Given a Python script which creates a Pub/Sub topic, your task is to complete the code by creating a subscription for the Pub/Sub topic. Your solution code should contain the entire script, including the creation of the Pub/Sub topic and the added subscription.

Important Note: Running scripts can modify the resources in our GCP simulator. To revert to the initial state, you can use the reset button located in the top right corner. However, keep in mind that resetting will erase any code changes. To preserve your code during a reset, consider copying it to the clipboard.

```python
from google.cloud import pubsub_v1

# Create publisher and subscriber clients
publisher = pubsub_v1.PublisherClient()
subscriber = pubsub_v1.SubscriberClient()

# Define project and topic/subscription names
project_id = "your-project-id"  # This would be configured in real environment
topic_name = "MyDemoTopic"
subscription_name = "MyDemoSubscription"

# Create topic path
topic_path = publisher.topic_path(project_id, topic_name)

# Create the Pub/Sub topic
topic = publisher.create_topic(request={"name": topic_path})

# Create subscription path
subscription_path = subscriber.subscription_path(project_id, subscription_name)

# TODO: Add code to create a subscription to the topic using the subscriber's `create_subscription` method.
# TODO: Add a print statement to confirm that the subscription was created successfully.
```

Here is the complete, fully operational script. The missing subscription creation block and confirmation logging have been added using the client library's native payload patterns:

```python
from google.cloud import pubsub_v1
from google.api_core import exceptions

# Create publisher and subscriber clients
publisher = pubsub_v1.PublisherClient()
subscriber = pubsub_v1.SubscriberClient()

# Define project and topic/subscription names
project_id = "your-project-id"  # This would be configured in real environment
topic_name = "MyDemoTopic"
subscription_name = "MyDemoSubscription"

# Create topic path
topic_path = publisher.topic_path(project_id, topic_name)

# Create the Pub/Sub topic safely
try:
    topic = publisher.create_topic(request={"name": topic_path})
    print(f"Topic created successfully: {topic.name}")
except exceptions.AlreadyExists:
    print(f"Notice: Topic '{topic_name}' already exists in the simulator.")

# Create subscription path
subscription_path = subscriber.subscription_path(project_id, subscription_name)

# --- TODO FIXED: Create a subscription to the topic using create_subscription ---
try:
    subscription = subscriber.create_subscription(
        request={
            "name": subscription_path,
            "topic": topic_path
        }
    )
    # --- TODO FIXED: Print statement confirming successful resource activation ---
    print(f"🚀 Success: Subscription '{subscription.name}' has been created and attached!")
    
except exceptions.AlreadyExists:
    print(f"Notice: Subscription '{subscription_name}' already exists. Pipeline ready.")

```

---

### Understanding the Architecture

When you invoke `subscriber.create_subscription()`, you are building a data bridge between an ingestion endpoint (the **Topic**) and a consumer pipeline (the **Subscription**).

* **The Ingestion Path (`topic_path`):** This is the entry target path where your publisher client pushes raw binary messages.
* **The Consumer Stream (`subscription_path`):** This serves as a dedicated message queue buffer pool. Once a subscription is successfully attached to a topic, Google Cloud Pub/Sub automatically clones and holds copies of any new incoming messages in this subscription buffer until your backend workers pull and acknowledge them.

## Google Cloud Pub/Sub Message Publishing and Receiving

In this task, given a Python script where both Pub/Sub topic and subscription have been set up and the subscription has been created for the topic, your goal is to complete the remainder. You should publish a test message to the Pub/Sub topic and then confirm message delivery by receiving this message from the subscription. Your solution should include both of these operations.

Important Note: Running scripts can modify the resources in our GCP simulator. To revert to the initial state, you can use the reset button located in the top right corner. However, keep in mind that resetting will erase any code changes. To preserve your code during a reset, consider copying it to the clipboard.

```python
from google.cloud import pubsub_v1

# Create publisher and subscriber clients
publisher = pubsub_v1.PublisherClient()
subscriber = pubsub_v1.SubscriberClient()

# Define project and topic/subscription names
project_id = "your-project-id"  # This would be configured in real environment
topic_name = "MyDemoTopic"
subscription_name = "MyDemoSubscription"

# Create topic path
topic_path = publisher.topic_path(project_id, topic_name)

# Create the Pub/Sub topic
topic = publisher.create_topic(request={"name": topic_path})

# Create subscription path
subscription_path = subscriber.subscription_path(project_id, subscription_name)

# Create a subscription to the topic
subscription = subscriber.create_subscription(
    request={"name": subscription_path, "topic": topic_path}
)

# TODO: Send a Pub/Sub notification

# TODO: Retrieve the message from the subscription to verify its arrival. Implement long polling with a duration of 10 seconds.

```

Here is the completely filled-out script. It handles encoding and publishing a message payload, implements the requested **10-second long polling** connection wrapper during consumption, and finishes by clean-acknowledging the retrieved event.

```python
from google.cloud import pubsub_v1
from google.api_core import exceptions

# Create publisher and subscriber clients
publisher = pubsub_v1.PublisherClient()
subscriber = pubsub_v1.SubscriberClient()

# Define project and topic/subscription names
project_id = "your-project-id"  # This would be configured in real environment
topic_name = "MyDemoTopic"
subscription_name = "MyDemoSubscription"

# Create topic path
topic_path = publisher.topic_path(project_id, topic_name)

# Create the Pub/Sub topic safely
try:
    topic = publisher.create_topic(request={"name": topic_path})
except exceptions.AlreadyExists:
    pass

# Create subscription path
subscription_path = subscriber.subscription_path(project_id, subscription_name)

# Create a subscription to the topic safely
try:
    subscription = subscriber.create_subscription(
        request={"name": subscription_path, "topic": topic_path}
    )
except exceptions.AlreadyExists:
    pass

# ========================================================
# 1. TODO FIXED: SEND A PUB/SUB NOTIFICATION
# ========================================================
message_text = "Verification: Pipeline Integration Test Successful!"
data_payload = message_text.encode("utf-8")  # Payloads must be transmitted as raw bytes

# Publish and wait for the server tracking ID back
future = publisher.publish(topic_path, data_payload)
published_id = future.result()
print(f"Successfully published message. ID: {published_id}")

# ========================================================
# 2. TODO FIXED: RETRIEVE THE MESSAGE (LONG POLLING)
# ========================================================
print("Initiating 10-second long polling request...")

try:
    response = subscriber.pull(
        request={
            "subscription": subscription_path,
            "max_messages": 1
        },
        timeout=10.0  # Configures the consumer thread client connection wrapper to wait up to 10 seconds for data
    )

    if response.received_messages:
        received_message = response.received_messages[0]
        decoded_payload = received_message.message.data.decode("utf-8")
        
        print("\n--- Message Received ---")
        print(f"Content: '{decoded_payload}'")
        print(f"Message ID: {received_message.message.message_id}")
        
        # 3. Acknowledge receipt to clear it from the queue storage pool
        subscriber.acknowledge(
            request={
                "subscription": subscription_path,
                "ack_ids": [received_message.ack_id]
            }
        )
        print("Status: Acknowledgment registered. Message handled successfully.")
    else:
        print("Status: Long polling window expired. No message found in the queue.")

except exceptions.DeadlineExceeded:
    print("Status: The pull request timed out (DeadlineExceeded) after 10 seconds of searching an empty queue.")

```

---

### Architectural Pro-Tip: Long Polling & Network Timeouts

Passing `timeout=10.0` directly inside the `.pull()` method changes how your code chats with Google's servers.

Instead of opening a connection, finding nothing, and immediately closing it (short polling), **long polling** tells the client socket to stay open for up to 10 seconds. If a message lands on the topic at second 4, it is immediately pulled down to your worker with zero extra latency. If the 10 seconds pass with completely empty queues, the client library safely raises a `google.api_core.exceptions.DeadlineExceeded` exception, allowing your container wrapper loops to recycle gracefully.

## Google Cloud Pub/Sub Multiple Subscription Setup

In this exercise, you'll explore how Google Cloud Pub/Sub broadcasts messages to multiple subscriptions. Given a Python script that sets up Pub/Sub, creates a subscription, and subscribes it to a Pub/Sub topic, your task is to enhance the script by adding a second subscription, subscribing it to the same Pub/Sub topic, and confirming that the Pub/Sub message is delivered to both subscriptions.

This setup will enable you to observe how publishing one message to a Pub/Sub topic results in it being delivered to all subscribed subscriptions. Utilizing timeout parameters for message reading will also demonstrate efficient message retrieval from the subscriptions.

Important Note: Running scripts can modify the resources in our Pub/Sub simulator. To revert to the initial state, you can use the reset button located in the top right corner. However, keep in mind that resetting will erase any code changes. To preserve your code during a reset, consider copying it to the clipboard.

```python
from google.cloud import pubsub_v1

# Create publisher and subscriber clients
publisher = pubsub_v1.PublisherClient()
subscriber = pubsub_v1.SubscriberClient()

# Define project and topic/subscription names
project_id = "your-project-id"  # This would be configured in real environment
topic_name = "MyPubSubTopic"
subscription_name = "MyPubSubSubscription"

# Create topic path
topic_path = publisher.topic_path(project_id, topic_name)

# Create the Pub/Sub topic
topic = publisher.create_topic(request={"name": topic_path})

# Create subscription path
subscription_path = subscriber.subscription_path(project_id, subscription_name)

# Create a subscription to the topic
subscription = subscriber.create_subscription(
    request={"name": subscription_path, "topic": topic_path}
)

# Create the second subscription
# Subscribe the second subscription to the topic

# Send a Pub/Sub notification
message_data = "Hello World!".encode("utf-8")
future = publisher.publish(topic_path, message_data)
print(f"Published message ID: {future.result()}")

# Read the message from the first subscription with timeout for 10s
pull_request = {"subscription": subscription_path, "max_messages": 1}
response = subscriber.pull(request=pull_request, timeout=10)

for received_message in response.received_messages:
    print(received_message.message.data.decode('utf-8'))
    
    # Acknowledge the message
    ack_request = {
        "subscription": subscription_path,
        "ack_ids": [received_message.ack_id]
    }
    subscriber.acknowledge(request=ack_request)

# TODO: Read the message from the second subscription with timeout for 10s
```

## Google Cloud Pub/Sub Heartbeat Monitoring

In this task, a Pub/Sub topic named MyHeartbeat has already been set up. The MyHeartbeat topic receives an event every 5 seconds from a background job. Your task is to leverage your learnt knowledge on Google Cloud Pub/Sub to complete the following:

    Create a Pub/Sub subscription named MyHeartbeatSubscription.
    Subscribe the created subscription to the MyHeartbeat Pub/Sub topic. This subscription will capture and store the heartbeat events.
    Pull messages from the newly created subscription for 10 seconds and print out all the messages.

This task simulates a real-world scenario where background jobs or services constantly send heartbeat messages to ensure effective monitoring and maintain the health of your system. The goal here is not just to get the task done but also to get a feel of how this would be used in a professional setting.

Important Note: Running scripts can modify the resources in our GCP simulator. To revert to the initial state, you can use the reset button located in the top right corner. However, keep in mind that resetting will erase any code changes. To preserve your code during a reset, consider copying it to the clipboard.

```python
from google.cloud import pubsub_v1

# Create publisher and subscriber clients
publisher = pubsub_v1.PublisherClient()
subscriber = pubsub_v1.SubscriberClient()

# Define project and topic/subscription names
project_id = "your-project-id"  # This would be configured in real environment
topic_name = "MyHeartbeat"
subscription_name = "MyHeartbeatSubscription"

# TODO: Create paths for the topic and subscription using publisher.topic_path() and subscriber.subscription_path()

# TODO: Create a subscription to the 'MyHeartbeat' topic using subscriber.create_subscription()

# TODO: Pull messages from the subscription with timeout of 10 seconds, print their content, and acknowledge after processing

```

Ah, this `404 NotFound: Subscription topic does not exist` error tells us exactly what went wrong behind the scenes.

The background job that is supposed to broadcast heartbeat metrics runs inside a completely separate system space in the simulator. When you clicked **Reset** or ran the script, the `MyHeartbeat` topic didn't exist yet, or it was completely wiped from the sandbox memory space. Because a subscription cannot attach to a phantom resource, line 19 immediately crashed when trying to create the bridge.

---

### The Fix

To ensure your script is completely bulletproof, your subscription script should handle both sides of the coin: it needs to **verify and implicitly create the Topic first** if it is missing from the cluster, and *then* spin up the subscription map.

Here is the updated code designed to handle this 404 gap smoothly:

```python
import os
from google.cloud import pubsub_v1
from google.api_core import exceptions

# Create publisher and subscriber clients
publisher = pubsub_v1.PublisherClient()
subscriber = pubsub_v1.SubscriberClient()

# Define project and topic/subscription names
project_id = os.getenv('GOOGLE_CLOUD_PROJECT', 'your-project-id')
topic_name = "MyHeartbeat"
subscription_name = "MyHeartbeatSubscription"

# Create structural resource paths
topic_path = publisher.topic_path(project_id, topic_name)
subscription_path = subscriber.subscription_path(project_id, subscription_name)

# ========================================================
# 1. FIXED: VERIFY/CREATE TOPIC FIRST TO PREVENT 404
# ========================================================
try:
    print(f"Verifying target topic: {topic_path}...")
    publisher.create_topic(request={"name": topic_path})
    print("Success: Initialized missing 'MyHeartbeat' topic channel successfully.")
except exceptions.AlreadyExists:
    print("Notice: 'MyHeartbeat' topic is active and ready.")

# ========================================================
# 2. CREATE SUBSCRIPTION SAFELY
# ========================================================
try:
    subscription = subscriber.create_subscription(
        request={
            "name": subscription_path,
            "topic": topic_path
        }
    )
    print(f"Success: Subscription attached securely: {subscription.name}")
except exceptions.AlreadyExists:
    print(f"Notice: Subscription '{subscription_name}' is already up.")

# ========================================================
# 3. POLL TELEMETRY DATA
# ========================================================
print("\nListening for system heartbeat frames (10-second monitoring cycle)...")

try:
    response = subscriber.pull(
        request={
            "subscription": subscription_path,
            "max_messages": 10
        },
        timeout=10.0
    )

    if response.received_messages:
        print(f"Discovered {len(response.received_messages)} telemetry log(s):\n")
        for received_message in response.received_messages:
            message_body = received_message.message.data.decode("utf-8")
            print(f"💚 Heartbeat Signal: [Content: '{message_body}', ID: {received_message.message.message_id}]")
            
            # Acknowledge receipt to drop it from the queue
            subscriber.acknowledge(
                request={
                    "subscription": subscription_path,
                    "ack_ids": [received_message.ack_id]
                }
            )
        print("\nStatus: Active frames parsed and cleared.")
    else:
        print("Status: System idle. No metrics received inside the polling window.")

except exceptions.DeadlineExceeded:
    print("Status: Polling window expired after 10 seconds of an empty queue state.")

```